In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from IPython.display import display

# Business Strategy and Conclusions

## 1. Decision Layer Over Segments
This notebook converts statistically validated clusters into action policies and measurable business decisions.

**Objectives:**
- Translate segment-level feature signatures into strategy archetypes.
- Define KPI uplift in a quantitative framework.
- Prioritize implementation using expected impact and operational constraints.

**Mathematical roadmap:**
Given segments $C_j$ with prevalence $p_j$, feature-level lifts $L_{j,f}$, and policy function $\pi_j$, we optimize decision value under constraints:

$$\pi_j: C_j \rightarrow \text{actions}, \qquad \Delta \mathrm{KPI}_{j} = \mathrm{KPI}_{j,\text{treat}} - \mathrm{KPI}_{j,\text{base}}$$

$$\max \sum_{j=1}^{k} p_j \cdot V_j(\pi_j) \quad \text{s.t. risk and budget constraints.}$$

## 2. Data Inputs from Notebook 4
We load the profiling artifacts produced previously:
- cluster prevalence table $p_j$,
- relative lift matrix $L_{j,f}$,
- ANOVA evidence and effect sizes $\eta_f^2$,
- labeled customer data for optional recalculation.

To keep execution robust across working directories, paths are resolved from common candidate locations.

In [2]:
def first_existing_path(candidates):
    path = next((Path(p) for p in candidates if Path(p).exists()), None)
    if path is None:
        raise FileNotFoundError(f'None of these paths exist: {candidates}')
    return path

data_with_clusters_path = first_existing_path([
    'data_with_clusters.csv',
    'notebooks/data_with_clusters.csv',
    '../data/data_with_clusters.csv'
 ])

df_segmented = pd.read_csv(data_with_clusters_path)

cluster_size_path = next((Path(p) for p in [
    'cluster_size_table.csv',
    'notebooks/cluster_size_table.csv'
 ] if Path(p).exists()), None)

lift_matrix_path = next((Path(p) for p in [
    'cluster_lift_matrix.csv',
    'notebooks/cluster_lift_matrix.csv'
 ] if Path(p).exists()), None)

anova_path = next((Path(p) for p in [
    'cluster_feature_anova.csv',
    'notebooks/cluster_feature_anova.csv'
 ] if Path(p).exists()), None)

if cluster_size_path is not None:
    cluster_size_table = pd.read_csv(cluster_size_path, index_col=0)
else:
    cluster_size_table = (
        df_segmented['Cluster'].value_counts().sort_index().rename_axis('Cluster').to_frame('Count')
    )
    cluster_size_table['Proportion'] = cluster_size_table['Count'] / len(df_segmented)

if lift_matrix_path is not None:
    lift_matrix = pd.read_csv(lift_matrix_path, index_col=0)
else:
    feature_cols = [col for col in df_segmented.columns if col not in ['Cluster', 'PC1', 'PC2']]
    global_mean = df_segmented[feature_cols].mean()
    denominator = global_mean.replace(0, np.nan)
    lift_matrix = (df_segmented.groupby('Cluster')[feature_cols].mean() - global_mean) / denominator

if anova_path is not None:
    anova_results = pd.read_csv(anova_path)
else:
    anova_results = pd.DataFrame(columns=['Feature', 'F-statistic', 'p-value', 'eta_squared', 'Significant (alpha=0.05)'])

cluster_size_table.index = cluster_size_table.index.astype(int)
lift_matrix.index = lift_matrix.index.astype(int)

print('Loaded data table shapes:')
print('df_segmented:', df_segmented.shape)
print('cluster_size_table:', cluster_size_table.shape)
print('lift_matrix:', lift_matrix.shape)
print('anova_results:', anova_results.shape)

display(cluster_size_table)

Loaded data table shapes:
df_segmented: (8950, 3)
cluster_size_table: (3, 2)
lift_matrix: (3, 17)
anova_results: (17, 5)


,Count,Proportion
Cluster,,
0,3167,0.353855
1,2939,0.328380
2,2844,0.317765


## 3. Strategy Scoring in Segment Space
We construct segment-level latent scores from feature lifts to guide policy decisions.

Define composite indices:
- Spend Intensity: average lift in purchase-volume features.
- Credit Stress: average lift in cash-advance and minimum-payment pressure.
- Payment Discipline: lift in full-payment behavior minus minimum-payment dependence.

These are not predictive models; they are interpretable linear summaries for strategic prioritization.

In [3]:
def safe_mean(row, columns):
    available = [c for c in columns if c in row.index]
    return row[available].mean() if available else np.nan

spend_cols = ['PURCHASES', 'ONEOFF_PURCHASES', 'INSTALLMENTS_PURCHASES', 'PURCHASES_TRX']
stress_cols = ['CASH_ADVANCE', 'CASH_ADVANCE_FREQUENCY', 'MINIMUM_PAYMENTS']

segment_scores = pd.DataFrame(index=lift_matrix.index)
segment_scores['Prevalence'] = cluster_size_table.reindex(lift_matrix.index)['Proportion']
segment_scores['Spend_Intensity'] = lift_matrix.apply(lambda r: safe_mean(r, spend_cols), axis=1)
segment_scores['Credit_Stress'] = lift_matrix.apply(lambda r: safe_mean(r, stress_cols), axis=1)

if {'PRC_FULL_PAYMENT', 'MINIMUM_PAYMENTS'}.issubset(set(lift_matrix.columns)):
    segment_scores['Payment_Discipline'] = lift_matrix['PRC_FULL_PAYMENT'] - lift_matrix['MINIMUM_PAYMENTS']
else:
    segment_scores['Payment_Discipline'] = np.nan

segment_scores['Growth_Opportunity'] = segment_scores['Spend_Intensity'] * segment_scores['Prevalence']
segment_scores['Risk_Priority'] = segment_scores['Credit_Stress'] * segment_scores['Prevalence']

def assign_archetype(row):
    if row['Credit_Stress'] > 0.4 and row['Payment_Discipline'] < 0:
        return 'Risk-Managed Revolvers'
    if row['Spend_Intensity'] > 0.5 and row['Payment_Discipline'] >= 0:
        return 'Premium Transactors'
    if row['Spend_Intensity'] < -0.2 and row['Credit_Stress'] < 0:
        return 'Low-Engagement Basics'
    return 'Balanced Mainstream'

segment_scores['Archetype'] = segment_scores.apply(assign_archetype, axis=1)
segment_scores = segment_scores.sort_values('Prevalence', ascending=False)

display(segment_scores.round(3))

,Prevalence,Spend_Intensity,Credit_Stress,Payment_Discipline,Growth_Opportunity,Risk_Priority,Archetype
Cluster,,,,,,,
0,0.354,1.279,0.028,-0.381,0.452,0.010,Balanced Mainstream
1,0.328,-0.493,-0.877,1.435,-0.162,-0.288,Low-Engagement Basics
2,0.318,-0.914,0.875,-1.059,-0.291,0.278,Risk-Managed Revolvers


## 4. KPI Uplift Framework
For each segment $j$, we evaluate policy quality through expected KPI uplift:

$$\Delta \mathrm{KPI}_{j} = \mathrm{KPI}_{j,\text{treat}} - \mathrm{KPI}_{j,\text{base}}.$$

To prioritize interventions, we use a prevalence-weighted proxy:

$$\mathrm{Priority}_{j} = p_j \cdot \left(w_g \cdot \mathrm{GrowthOpportunity}_j + w_r \cdot \mathrm{RiskPriority}_j\right),$$

with weights $(w_g, w_r)$ reflecting strategic emphasis (growth vs. risk control).

In [4]:
w_growth = 0.6
w_risk = 0.4

kpi_plan = segment_scores.copy()
kpi_plan['Priority_Score'] = (
    kpi_plan['Prevalence']
    * (w_growth * kpi_plan['Growth_Opportunity'].abs() + w_risk * kpi_plan['Risk_Priority'].abs())
)

kpi_plan = kpi_plan.sort_values('Priority_Score', ascending=False)

display(kpi_plan[['Archetype', 'Prevalence', 'Growth_Opportunity', 'Risk_Priority', 'Priority_Score']].round(4))

,Archetype,Prevalence,Growth_Opportunity,Risk_Priority,Priority_Score
Cluster,,,,,
0,Balanced Mainstream,0.3539,0.4524,0.0098,0.0974
2,Risk-Managed Revolvers,0.3178,-0.2906,0.2782,0.0908
1,Low-Engagement Basics,0.3284,-0.1619,-0.2880,0.0697


## 5. Policy Matrix by Segment
We map each archetype to an operational policy tuple
$$\pi_j = (\text{offer}, \text{channel}, \text{cadence}, \text{risk guardrail}).$$

This ensures strategy is not only descriptive but directly executable in CRM and campaign systems.

In [5]:
policy_lookup = {
    'Premium Transactors': {
        'Primary Offer': 'Premium rewards and lifestyle bundles',
        'Channel': 'App + email',
        'Cadence': 'Weekly personalized recommendations',
        'Risk Guardrail': 'Monitor utilization spike > 20%'
    },
    'Risk-Managed Revolvers': {
        'Primary Offer': 'Debt consolidation and payment-plan nudges',
        'Channel': 'SMS + call center',
        'Cadence': 'Twice-weekly repayment prompts',
        'Risk Guardrail': 'Trigger intervention at delinquency probability threshold'
    },
    'Low-Engagement Basics': {
        'Primary Offer': 'Activation cashback and low-friction onboarding',
        'Channel': 'Push notifications',
        'Cadence': 'Bi-weekly habit-building prompts',
        'Risk Guardrail': 'Cap incentive spend per active user'
    },
    'Balanced Mainstream': {
        'Primary Offer': 'Cross-sell card upgrades and installment options',
        'Channel': 'Email + in-app cards',
        'Cadence': 'Monthly campaign cycles',
        'Risk Guardrail': 'Exclude users with adverse recent payment trend'
    }
}

policy_rows = []
for cluster_id, row in kpi_plan.iterrows():
    archetype = row['Archetype']
    policy = policy_lookup.get(archetype, policy_lookup['Balanced Mainstream'])
    policy_rows.append({
        'Cluster': cluster_id,
        'Archetype': archetype,
        'Priority_Score': row['Priority_Score'],
        'Primary Offer': policy['Primary Offer'],
        'Channel': policy['Channel'],
        'Cadence': policy['Cadence'],
        'Risk Guardrail': policy['Risk Guardrail']
    })

policy_matrix = pd.DataFrame(policy_rows).sort_values('Priority_Score', ascending=False)
display(policy_matrix)

,Cluster,Archetype,Priority_Score,Primary Offer,Channel,Cadence,Risk Guardrail
0,0,Balanced Mainstream,0.097444,Cross-sell card upgrades and installment options,Email + in-app cards,Monthly campaign cycles,Exclude users with adverse recent payment trend
1,2,Risk-Managed Revolvers,0.090757,Debt consolidation and payment-plan nudges,SMS + call center,Twice-weekly repayment prompts,Trigger intervention at delinquency probabilit...
2,1,Low-Engagement Basics,0.069720,Activation cashback and low-friction onboarding,Push notifications,Bi-weekly habit-building prompts,Cap incentive spend per active user


## 6. Conclusions and Project Closure
This notebook completed the transition from unsupervised structure discovery to decision policy design.

From Notebook 2 to Notebook 5, the pipeline is:
1. preprocess and stabilize geometry,
2. reduce dimensionality and identify cluster structure,
3. profile cluster economics and validate separability,
4. map segments to constrained action policies and prioritized rollout.

The resulting deliverables are segment-level priority rankings, policy matrix definitions, and exportable tables for implementation.

In [6]:
kpi_plan.to_csv('segment_kpi_priority_table.csv')
policy_matrix.to_csv('segment_policy_matrix.csv', index=False)

print('Exported: segment_kpi_priority_table.csv, segment_policy_matrix.csv')

Exported: segment_kpi_priority_table.csv, segment_policy_matrix.csv


## What We Achieved

In simple terms, we turned a large customer table into clear groups and then decided what action to take for each group.

Here is what was achieved end-to-end:
1. We cleaned and transformed the raw data so the patterns were mathematically stable.
2. We found natural customer segments using clustering in PCA space.
3. We translated those segments back into real financial behavior (spending, balance, cash-advance habits).
4. We measured which differences are statistically meaningful, not only visually different.
5. We created a practical action plan: which segment to prioritize, what offer to use, and what risk rule to keep.

Business outcome: this project gives a reproducible segmentation system that supports targeted campaigns, better allocation of marketing effort, and safer growth decisions.